# CONFIGURATION

This script creates an interactive NU-WRF vs Ameriflux dashboard. How it works:
 - use the state filter dropdown menu  to visualize all of CONUS or individual states.
 - Click on a station to see the time series and performance statistics of the NU-WRF data.
 - The Ameriflux data availability is shown per station, to help understand the data.
 - The colors of the stations indicates the IGBP code, which will also be provided when a station is selected.

In [1]:
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import netCDF4 as nc
from scipy.spatial import cKDTree
import glob, os
from datetime import datetime, timedelta
from tqdm import tqdm
import json
import xarray as xr
import pickle

In [2]:
# ─── CONFIG ──────────────────────────────────────────────────────────────────
# PATH_AMERIFLUX = "/dodrio/scratch/projects/2022_200/project_output/rsda/vsc33651/AmeriFlux"
PATH_AMERIFLUX = "/staging/leuven/stg_00024/OUTPUT/jonasm/Ameriflux"
PATH_WRF = "/staging/leuven/stg_00024/OUTPUT/jonasm/METEORI/lambert/test_output"
WRF_START = "2015-05-01"
WRF_END   = "2015-07-17"
EXCLUDE_SITES = ['US-Tw1', 'US-Myb', 'US-Sne', 'US-Var']
WRF_VARS = ["LH", "HFX", "T2", "SWDOWN", "Q2"]

IGBP_COLORS = {
    'ENF': '#97C1A9', 'EBF': '#97C1A9', 'DNF': '#97C1A9',
    'DBF': '#97C1A9', 'MF': '#97C1A9', 'CSH': '#CCCCEE',
    'OSH': '#CCCCEE', 'WSA': '#FEB7B1', 'SAV': '#FEB7B1',
    'GRA': '#C5E2E8', 'WET': '#7AB8B8', 'CRO': '#FFFFCB',
    'CVM': '#FFFFCB', 'URB': '#A9A9A9', 'SNO': '#F5F5FF',
    'BSV': '#EAEAEA', 'WAT': '#FFFFFF', 'Unknown': '#888888',
    }

# (wrf_col, amf_col, label, units)
VAR_MAP = [
    ("LH",    "LE_F_MDS", "Latent Heat",   "W m⁻²"),
    ("HFX",   "H_F_MDS",  "Sensible Heat", "W m⁻²"),
    ("T2_C",  "TA_F",     "Air Temp 2m",   "°C"),
    ("SWDOWN","SW_IN_F",  "SW Radiation",  "W m⁻²"),
]

# FUNCTIONS

In [3]:
def pearson_r(obs, sim):
    mask = np.isfinite(obs) & np.isfinite(sim)
    if mask.sum() < 10: return np.nan
    o, s = obs[mask], sim[mask]
    return float(np.corrcoef(o, s)[0, 1])

def ubrmsd(obs, sim):
    mask = np.isfinite(obs) & np.isfinite(sim)
    if mask.sum() < 10: return np.nan
    o, s = obs[mask], sim[mask]
    return float(np.sqrt(np.mean(((s - s.mean()) - (o - o.mean()))**2)))

def bias(obs, sim):
    mask = np.isfinite(obs) & np.isfinite(sim)
    if mask.sum() < 10: return np.nan
    return float(np.mean(sim[mask] - obs[mask]))

def sanitize(obj):
    if isinstance(obj, dict):  return {k: sanitize(v) for k, v in obj.items()}
    if isinstance(obj, list):  return [sanitize(v) for v in obj]
    if isinstance(obj, float) and (np.isnan(obj) or np.isinf(obj)): return None
    return obj

# CODE

## LOAD SITE COORDS

In [4]:
# ─── LOAD ALL SITE COORDS ─────────────────────────────────────────────────────
print("Loading site coordinates...")
coords = pd.read_csv(os.path.join(PATH_AMERIFLUX, "site_coords.csv"))
coords = coords[~coords['SITE_ID'].isin(EXCLUDE_SITES)].copy()
coords['IGBP']  = coords['IGBP'].fillna('Unknown')
coords['STATE'] = coords['STATE'].fillna('Unknown')
coords['CROPLAND'] = coords['IGBP'] == 'CRO'

available_states = sorted(coords['STATE'].dropna().unique().tolist())
print(f"  {len(coords)} total sites across {len(available_states)} states")
print(f"  Availble states: {available_states}")

Loading site coordinates...
  137 total sites across 34 states
  Availble states: ['AL', 'AR', 'AZ', 'CA', 'CO', 'DE', 'FL', 'ID', 'IN', 'KS', 'LA', 'MA', 'MD', 'ME', 'MI', 'MN', 'MO', 'NC', 'ND', 'NE', 'NH', 'NM', 'OH', 'OK', 'OR', 'PA', 'SC', 'TN', 'TX', 'UT', 'VA', 'WA', 'WI', 'WY']


## NUWRF OUTPUT

In [5]:
# ─── LOAD WRF FILES ───────────────────────────────────────────────────────────
def load_wrf_files(domain):
    files = sorted(glob.glob(os.path.join(PATH_WRF, f"wrfout_{domain}_*")))
    valid = []
    for f in files:
        parts = os.path.basename(f).split('_')
        if len(parts) < 4: continue
        try:
            dt = datetime.strptime(f"{parts[2]} {parts[3]}", "%Y-%m-%d %H:%M:%S")
            if WRF_START <= dt.strftime("%Y-%m-%d") <= WRF_END:
                valid.append((f, dt))
        except: continue
    return valid

print("Scanning WRF files...")
d01_files = load_wrf_files("d01")
d02_files = load_wrf_files("d02")
print(f"  d01: {len(d01_files)} files  |  d02: {len(d02_files)} files")

Scanning WRF files...
  d01: 1852 files  |  d02: 1852 files


In [6]:
# ─── MATCH SITES TO GRIDS ────────────────────────────────────────────────────
def get_grid_and_match(files, sites):
    if not files: return None, None, None
    with nc.Dataset(files[0][0]) as ds:
        glat = ds.variables['XLAT'][0, :, :]
        glon = ds.variables['XLONG'][0, :, :]
    tree = cKDTree(np.column_stack([glat.ravel(), glon.ravel()]))
    _, idxs = tree.query(np.column_stack([sites['LOCATION_LAT'].values,
                                           sites['LOCATION_LONG'].values]))
    rows, cols = np.unravel_index(idxs, glat.shape)
    return glat, glon, list(zip(rows.tolist(), cols.tolist()))

def sites_in_domain(glat, glon, sites):
    if glat is None: return [False] * len(sites)
    return [(glat.min() <= r['LOCATION_LAT'] <= glat.max() and
             glon.min() <= r['LOCATION_LONG'] <= glon.max())
            for _, r in sites.iterrows()]

print("Matching ALL sites to WRF grids...")
d01_lat, d01_lon, d01_ij = get_grid_and_match(d01_files, coords) if d01_files else (None, None, None)
d02_lat, d02_lon, d02_ij = get_grid_and_match(d02_files, coords) if d02_files else (None, None, None)
coords['IN_D02'] = sites_in_domain(d02_lat, d02_lon, coords)
coords_in_wrf = coords[coords['IN_D02'] | (d01_lat is not None)].copy()
print(f"  Sites in d01 domain: {len(coords)}  |  Sites in d02: {coords['IN_D02'].sum()}")

Matching ALL sites to WRF grids...
  Sites in d01 domain: 137  |  Sites in d02: 47


In [7]:
# ─── EXTRACT WRF TIME SERIES ─────────────────────────────────────────────────
def extract_wrf_ts(files, ij_pairs, sites):
    if not files or ij_pairs is None:
        return {}

    utc_map = sites.set_index('SITE_ID')['UTC_OFFSET'].to_dict()
    site_list = [(sid, ij_pairs[k][0], ij_pairs[k][1], utc_map.get(sid, 0))
                 for k, (_, site) in enumerate(sites.iterrows())
                 for sid in [site['SITE_ID']]]

    # Pre-allocate: dict of sid -> {var -> list}
    records = {sid: {'time': [], **{v: [] for v in WRF_VARS}} for sid, *_ in site_list}

    for fpath, file_time_utc in tqdm(files, desc="  Reading WRF"):
        with nc.Dataset(fpath) as ds:
            # Read each variable once into memory, then index all sites
            var_arrays = {}
            for var in WRF_VARS:
                if var in ds.variables:
                    arr = np.squeeze(ds.variables[var][:])
                    var_arrays[var] = arr
            
            for sid, i, j, offset in site_list:
                records[sid]['time'].append(file_time_utc + timedelta(hours=offset))
                for var in WRF_VARS:
                    if var in var_arrays:
                        arr = var_arrays[var]
                        val = arr[i, j] if arr.ndim == 2 else arr[0, i, j]
                        records[sid][var].append(float(val) if not np.ma.is_masked(val) else np.nan)
                    else:
                        records[sid][var].append(np.nan)

    result = {}
    for sid, i, j, offset in site_list:
        df = pd.DataFrame(records[sid]).set_index('time')
        df.index = pd.to_datetime(df.index)
        if 'T2' in df.columns:
            df['T2_C'] = df['T2'] - 273.15
        result[sid] = df
    return result

In [8]:
CACHE_PATH = "./wrf_ts_cache.pkl"

if os.path.exists(CACHE_PATH):
    print("Loading WRF time series from cache...")
    with open(CACHE_PATH, "rb") as f:
        d01_ts, d02_ts = pickle.load(f)
    print(f"  Loaded d01: {len(d01_ts)} sites, d02: {len(d02_ts)} sites")
else:
    print("Extracting d01 time series...")
    d01_ts = extract_wrf_ts(d01_files, d01_ij, coords) if d01_files else {}
    print("Extracting d02 time series...")
    d02_ts = extract_wrf_ts(d02_files, d02_ij, coords) if d02_files else {}
    with open(CACHE_PATH, "wb") as f:
        pickle.dump((d01_ts, d02_ts), f)
    print(f"  Cache saved to {CACHE_PATH}")

Loading WRF time series from cache...
  Loaded d01: 137 sites, d02: 137 sites


## LOAD AMERIFLUX

In [11]:
# ─── LOAD ALL AMERIFLUX ───────────────────────────────────────────────────────
print("Loading AmeriFlux data (all sites)...")
all_site_ids = set(coords['SITE_ID'])
amf_ts = {}
amf_coverage = {}  # sid -> (start_str, end_str, wrf_overlap_hrs)

for fpath in tqdm(glob.glob(os.path.join(PATH_AMERIFLUX, "extracted/*.csv"))):
    fname = os.path.basename(fpath)
    if "AMF_" not in fname or "_FLUXNET" not in fname: continue
    sid = fname.split('_')[1]
    if sid not in all_site_ids: continue
    try:
        df = pd.read_csv(fpath, parse_dates=["TIMESTAMP_START"],
                         date_format="%Y%m%d%H%M", low_memory=False)
        df = df.rename(columns={"TIMESTAMP_START": "time"}).set_index("time")
        df = df.replace(-9999.0, np.nan)
        for var, qcvar in [("LE_F_MDS","LE_F_MDS_QC"),("H_F_MDS","H_F_MDS_QC"),("SW_IN_F","SW_IN_F_QC")]:
            if var in df.columns and qcvar in df.columns:
                df.loc[df[qcvar] > 1, var] = np.nan
        keep = ["LE_F_MDS", "H_F_MDS", "TA_F", "SW_IN_F", "P_F"]
        df = df[[c for c in keep if c in df.columns]].resample("1h").mean()
        amf_ts[sid] = df

        # Record full coverage and overlap with WRF window
        full_start = str(df.index.min().date())
        full_end   = str(df.index.max().date())
        overlap = df[(df.index >= WRF_START) & (df.index <= WRF_END)]
        non_nan = int(overlap['LE_F_MDS'].notna().sum()) if 'LE_F_MDS' in overlap.columns else 0
        amf_coverage[sid] = {
            'full_start': full_start,
            'full_end':   full_end,
            'overlap_hrs': int(len(overlap)),
            'valid_le_hrs': non_nan,
        }
    except Exception as e:
        print(f"  Warning {sid}: {e}")

print(f"  Loaded {len(amf_ts)} AmeriFlux sites")

Loading AmeriFlux data (all sites)...


100%|██████████| 137/137 [12:04<00:00,  5.29s/it]

  Loaded 133 AmeriFlux sites


## MERGE DATASETS

In [12]:
# ─── MERGE & STATS ────────────────────────────────────────────────────────────
CACHE_PATH = "./wrf_ameriflux_cache.pkl"

if os.path.exists(CACHE_PATH):
    print(f"Loading merged data from cache {CACHE_PATH}...")
    with open(CACHE_PATH, "rb") as f:
        merged_d01, merged_d02, site_data = pickle.load(f)
    print(f"  Loaded {len(site_data)} sites from cache")

else:
    print("Merging and computing statistics...")
    merged_d01, merged_d02 = {}, {}
    site_data = {}

    for _, site in coords.iterrows():
        sid = site['SITE_ID']

        cov = amf_coverage.get(sid, {'full_start': None, 'full_end': None,
                                      'overlap_hrs': 0, 'valid_le_hrs': 0})
        entry = {
            'sid':    sid,
            'igbp':   site['IGBP'],
            'state':  site['STATE'],
            'lat':    float(site['LOCATION_LAT']),
            'lon':    float(site['LOCATION_LONG']),
            'in_d02': bool(site['IN_D02']),
            'amf_full_start':  cov['full_start'],
            'amf_full_end':    cov['full_end'],
            'amf_overlap_hrs': cov['overlap_hrs'],
            'amf_valid_le':    cov['valid_le_hrs'],
            'has_amf':   sid in amf_ts,
            'stats':     {'d01': {}, 'd02': {}},
            'timeseries':{'d01': None, 'd02': None},
        }

        if sid in amf_ts:
            a = amf_ts[sid]
            a = a[(a.index >= WRF_START) & (a.index <= WRF_END)]

            for domain, ts_dict, mdict in [("d01", d01_ts, merged_d01),
                                            ("d02", d02_ts, merged_d02)]:
                if sid not in ts_dict: continue
                w = ts_dict[sid]
                w = w[(w.index >= WRF_START) & (w.index <= WRF_END)]
                if w.empty or a.empty: continue
                merged = w.join(a, how="inner", lsuffix="_wrf", rsuffix="_amf")
                mdict[sid] = merged

                for wvar, avar, label, units in VAR_MAP:
                    if wvar in merged.columns and avar in merged.columns:
                        o = merged[avar].values.astype(float)
                        s = merged[wvar].values.astype(float)
                        entry['stats'][domain][wvar] = {
                            'label': label, 'units': units,
                            'r':      pearson_r(o, s),
                            'ubrmsd': ubrmsd(o, s),
                            'bias':   bias(o, s),
                        }

                m = mdict.get(sid)
                if m is not None:
                    ts = {'time': [str(t) for t in m.index]}
                    for wvar, avar, label, units in VAR_MAP:
                        ts[f'wrf_{wvar}'] = [None if (v is None or np.isnan(v)) else round(float(v), 3)
                                              for v in (m[wvar].values if wvar in m.columns else [np.nan]*len(m))]
                        ts[f'amf_{avar}'] = [None if (v is None or np.isnan(v)) else round(float(v), 3)
                                              for v in (m[avar].values if avar in m.columns else [np.nan]*len(m))]
                    entry['timeseries'][domain] = ts

        site_data[sid] = entry

    print(f"  Built site_data for {len(site_data)} stations")

    print("Saving cache...")
    with open(CACHE_PATH, "wb") as f:
        pickle.dump((merged_d01, merged_d02, site_data), f)
    print(f"  Cache saved to {CACHE_PATH}")

Loading merged data from cache ./wrf_ameriflux_cache.pkl...
  Loaded 137 sites from cache


In [13]:
# ─── MERGE & STATS ────────────────────────────────────────────────────────────
print("Merging and computing statistics...")
merged_d01, merged_d02 = {}, {}
site_data = {}

for _, site in coords.iterrows():
    sid = site['SITE_ID']

    # AmeriFlux coverage info (even if no WRF overlap)
    cov = amf_coverage.get(sid, {'full_start': None, 'full_end': None,
                                  'overlap_hrs': 0, 'valid_le_hrs': 0})

    entry = {
        'sid':    sid,
        'igbp':   site['IGBP'],
        'state':  site['STATE'],
        'lat':    float(site['LOCATION_LAT']),
        'lon':    float(site['LOCATION_LONG']),
        'in_d02': bool(site['IN_D02']),
        'amf_full_start':  cov['full_start'],
        'amf_full_end':    cov['full_end'],
        'amf_overlap_hrs': cov['overlap_hrs'],
        'amf_valid_le':    cov['valid_le_hrs'],
        'has_amf':   sid in amf_ts,
        'stats':     {'d01': {}, 'd02': {}},
        'timeseries':{'d01': None, 'd02': None},
    }

    if sid in amf_ts:
        a = amf_ts[sid]
        a = a[(a.index >= WRF_START) & (a.index <= WRF_END)]

        for domain, ts_dict, mdict in [("d01", d01_ts, merged_d01),
                                        ("d02", d02_ts, merged_d02)]:
            if sid not in ts_dict: continue
            w = ts_dict[sid]
            w = w[(w.index >= WRF_START) & (w.index <= WRF_END)]
            if w.empty or a.empty: continue
            merged = w.join(a, how="inner", lsuffix="_wrf", rsuffix="_amf")
            mdict[sid] = merged

            # Stats
            for wvar, avar, label, units in VAR_MAP:
                if wvar in merged.columns and avar in merged.columns:
                    o = merged[avar].values.astype(float)
                    s = merged[wvar].values.astype(float)
                    entry['stats'][domain][wvar] = {
                        'label': label, 'units': units,
                        'r':      pearson_r(o, s),
                        'ubrmsd': ubrmsd(o, s),
                        'bias':   bias(o, s),
                    }

            # Time series
            m = mdict.get(sid)
            if m is not None:
                ts = {'time': [str(t) for t in m.index]}
                for wvar, avar, label, units in VAR_MAP:
                    ts[f'wrf_{wvar}'] = [None if (v is None or np.isnan(v)) else round(float(v), 3)
                                          for v in (m[wvar].values if wvar in m.columns else [np.nan]*len(m))]
                    ts[f'amf_{avar}'] = [None if (v is None or np.isnan(v)) else round(float(v), 3)
                                          for v in (m[avar].values if avar in m.columns else [np.nan]*len(m))]
                entry['timeseries'][domain] = ts

    site_data[sid] = entry

print(f"  Built site_data for {len(site_data)} stations")

Merging and computing statistics...
  Built site_data for 137 stations


In [14]:
# ─── AGGREGATE STATS ──────────────────────────────────────────────────────────
def aggregate_stats(entries):
    """Given a list of site_data entries, pool obs/sim arrays and compute stats."""
    agg = {}
    for wvar, avar, label, units in VAR_MAP:
        for domain in ("d01", "d02"):
            obs_all, sim_all = [], []
            for e in entries:
                ts = e["timeseries"][domain]
                if ts is None:
                    continue
                obs = np.array(ts[f"amf_{avar}"], dtype=float)
                sim = np.array(ts[f"wrf_{wvar}"], dtype=float)
                obs_all.append(obs)
                sim_all.append(sim)
            if not obs_all:
                continue
            obs_cat = np.concatenate(obs_all)
            sim_cat = np.concatenate(sim_all)
            key = f"{domain}_{wvar}"
            agg[key] = {
                "label": label, "units": units, "domain": domain,
                "r":      pearson_r(obs_cat, sim_cat),
                "ubrmsd": ubrmsd(obs_cat, sim_cat),
                "bias":   bias(obs_cat, sim_cat),
                "n_sites": sum(1 for e in entries if e["timeseries"][domain] is not None),
            }
    return agg

In [15]:
# Build per-state and overall aggregates
all_entries = list(site_data.values())
agg_all = aggregate_stats(all_entries)

agg_by_state = {}
for state in available_states:
    state_entries = [e for e in all_entries if e["state"] == state]
    agg_by_state[state] = aggregate_stats(state_entries)

AGG_ALL_JSON      = json.dumps(sanitize(agg_all))
AGG_BY_STATE_JSON = json.dumps(sanitize(agg_by_state))

## DASHBOARD

In [18]:
# ─── MAP TRACES (all sites) ───────────────────────────────────────────────────
map_traces_data = []
for igbp, color in IGBP_COLORS.items():
    sub = coords[coords['IGBP'] == igbp]
    if sub.empty: continue
    map_traces_data.append({
        'type': 'scattermapbox',
        'lat':        sub['LOCATION_LAT'].tolist(),
        'lon':        sub['LOCATION_LONG'].tolist(),
        'mode':       'markers',
        'marker':     {'size': 12, 'color': color, 'opacity': 0.9},
        'name':       igbp,
        'text':       sub['SITE_ID'].tolist(),
        'customdata': sub['SITE_ID'].tolist(),
        'hovertemplate': (
            '<b>%{text}</b><br>'
            f'IGBP: {igbp}<br>'
            'Lat: %{lat:.3f}  Lon: %{lon:.3f}<br>'
            '<i>Click to explore</i><extra></extra>'
        ),
    })

map_center_lat = float(coords['LOCATION_LAT'].mean())
map_center_lon = float(coords['LOCATION_LONG'].mean())

# ─── SERIALISE ────────────────────────────────────────────────────────────────
site_data_json   = json.dumps(sanitize(site_data))
map_traces_json  = json.dumps(sanitize(map_traces_data))
VAR_MAP_JSON     = json.dumps([{'wvar':wv,'avar':av,'label':lb,'units':un} for wv,av,lb,un in VAR_MAP])
IGBP_COLORS_JSON = json.dumps(IGBP_COLORS)
STATES_JSON      = json.dumps(["All USA"] + available_states)
D01_BBOX_JSON    = json.dumps(sanitize(d01_bbox))
D02_BBOX_JSON    = json.dumps(sanitize(d02_bbox))

In [17]:
# ─── DOMAIN BOUNDING BOXES ───────────────────────────────────────────────────
def domain_bbox(glat, glon):
    """Return corner lats/lons of domain outline as closed polygon for Scattermapbox."""
    if glat is None or glon is None:
        return None
    # Use actual grid corners (all 4 edges) for a proper outline
    lats_top    = glat[0, :].tolist()
    lons_top    = glon[0, :].tolist()
    lats_right  = glat[:, -1].tolist()
    lons_right  = glon[:, -1].tolist()
    lats_bottom = glat[-1, ::-1].tolist()
    lons_bottom = glon[-1, ::-1].tolist()
    lats_left   = glat[::-1, 0].tolist()
    lons_left   = glon[::-1, 0].tolist()
    lats = lats_top + lats_right + lats_bottom + lats_left + [lats_top[0]]
    lons = lons_top + lons_right + lons_bottom + lons_left + [lons_top[0]]
    return {'lat': lats, 'lon': lons}

d01_bbox = domain_bbox(d01_lat, d01_lon)
d02_bbox = domain_bbox(d02_lat, d02_lon)

HTML code, made with CLAUDE.AI. Make sure to adjust the out_path at the end if needed.

In [20]:
# ─── HTML ─────────────────────────────────────────────────────────────────────
html = f"""<!DOCTYPE html>
<html lang="en">
<head>
<meta charset="UTF-8">
<meta name="viewport" content="width=device-width, initial-scale=1.0">
<title>NU-WRF × AmeriFlux</title>
<script src="https://cdn.plot.ly/plotly-2.32.0.min.js"></script>
<link href="https://fonts.googleapis.com/css2?family=IBM+Plex+Mono:wght@400;600&family=IBM+Plex+Sans:wght@300;400;600&display=swap" rel="stylesheet">
<style>
  :root {{
    --bg:#0e1117; --surface:#161b27; --surface2:#1e2535;
    --border:#2a3348; --accent:#4fc3f7; --accent2:#81c784;
    --text:#e8eaf0; --muted:#7a8499;
    --d01:#4fc3f7; --d02:#ffb74d; --obs:#81c784;
    --warn:#f28b30; --ok:#81c784; --none:#7a8499;
  }}
  * {{ box-sizing:border-box; margin:0; padding:0; }}
  body {{
    font-family:'IBM Plex Sans',sans-serif;
    background:var(--bg); color:var(--text);
    height:100vh; display:flex; flex-direction:column; overflow:hidden;
  }}
  header {{
    padding:10px 20px; border-bottom:1px solid var(--border);
    background:var(--surface); flex-shrink:0;
    display:flex; align-items:center; gap:20px; flex-wrap:wrap;
  }}
  header h1 {{
    font-family:'IBM Plex Mono',monospace; font-size:0.95rem;
    font-weight:600; color:var(--accent); letter-spacing:0.04em; white-space:nowrap;
  }}
  header span {{
    font-size:0.72rem; color:var(--muted);
    font-family:'IBM Plex Mono',monospace;
  }}
  .state-filter {{
    display:flex; align-items:center; gap:8px; margin-left:auto;
  }}
  .state-filter label {{
    font-family:'IBM Plex Mono',monospace; font-size:0.7rem;
    color:var(--muted); text-transform:uppercase; letter-spacing:0.06em;
  }}
  #state-select {{
    background:var(--surface2); border:1px solid var(--border);
    color:var(--text); font-family:'IBM Plex Mono',monospace;
    font-size:0.75rem; padding:4px 10px 4px 8px; border-radius:5px;
    cursor:pointer; outline:none; min-width:130px;
    appearance:none; -webkit-appearance:none;
    background-image:url("data:image/svg+xml,%3Csvg xmlns='http://www.w3.org/2000/svg' width='10' height='6'%3E%3Cpath d='M0 0l5 6 5-6z' fill='%237a8499'/%3E%3C/svg%3E");
    background-repeat:no-repeat; background-position:right 8px center;
    padding-right:24px;
  }}
  #state-select:focus {{ border-color:var(--accent); }}
  .domain-toggles {{
    display:flex; align-items:center; gap:6px;
  }}
  .domain-toggles label {{
    font-family:'IBM Plex Mono',monospace; font-size:0.7rem;
    color:var(--muted); text-transform:uppercase; letter-spacing:0.06em;
  }}
  .dom-btn {{
    background:var(--surface2); border:1px solid var(--border);
    font-family:'IBM Plex Mono',monospace; font-size:0.72rem; font-weight:600;
    padding:3px 11px; border-radius:4px; cursor:pointer; transition:all 0.15s;
    letter-spacing:0.04em;
  }}
  #toggle-d01 {{ color:var(--d01); }}
  #toggle-d01.active {{ background:rgba(79,195,247,0.12); border-color:var(--d01); }}
  #toggle-d02 {{ color:var(--d02); }}
  #toggle-d02.active {{ background:rgba(255,183,77,0.12); border-color:var(--d02); }}
  .dom-btn:not(.active) {{ opacity:0.35; }}
  #site-count {{
    font-family:'IBM Plex Mono',monospace; font-size:0.68rem;
    color:var(--muted); white-space:nowrap;
  }}
  .layout {{
    display:grid; grid-template-columns:1fr 1fr;
    grid-template-rows:1fr 1fr; flex:1; min-height:0;
  }}
  .panel {{
    border:1px solid var(--border); background:var(--surface);
    display:flex; flex-direction:column; min-height:0; overflow:hidden;
  }}
  .ph {{
    padding:8px 14px; border-bottom:1px solid var(--border);
    font-family:'IBM Plex Mono',monospace; font-size:0.65rem; font-weight:600;
    letter-spacing:0.08em; text-transform:uppercase; color:var(--muted);
    display:flex; align-items:center; gap:8px; flex-shrink:0;
  }}
  .ph .dot {{ width:7px; height:7px; border-radius:50%; background:var(--accent); flex-shrink:0; }}
  #map-panel {{ grid-column:1; grid-row:1/3; }}
  #stats-panel {{ grid-column:2; grid-row:1; overflow-y:auto; }}
  #ts-panel {{ grid-column:2; grid-row:2; }}
  #map-div {{ flex:1; min-height:0; }}

  /* ── stats ── */
  #site-info {{ padding:14px 16px; overflow-y:auto; flex:1; }}
  .site-name {{
    font-family:'IBM Plex Mono',monospace; font-size:1.1rem;
    font-weight:600; color:var(--accent); margin-bottom:3px;
  }}
  .site-meta {{
    font-size:0.72rem; color:var(--muted); margin-bottom:8px;
    font-family:'IBM Plex Mono',monospace; line-height:1.6;
  }}
  /* AmeriFlux coverage bar */
  .amf-coverage {{
    background:var(--surface2); border:1px solid var(--border);
    border-radius:6px; padding:9px 12px; margin-bottom:10px;
    font-family:'IBM Plex Mono',monospace; font-size:0.7rem;
  }}
  .amf-coverage .cov-title {{
    color:var(--muted); text-transform:uppercase; letter-spacing:0.07em;
    font-size:0.63rem; margin-bottom:6px;
  }}
  .cov-bar-wrap {{
    position:relative; height:10px; background:#1e2535;
    border-radius:3px; margin:4px 0 2px; overflow:hidden;
    border:1px solid var(--border);
  }}
  .cov-bar-full  {{ position:absolute; top:0; height:100%; background:#2a3348; border-radius:3px; }}
  .cov-bar-wrf   {{ position:absolute; top:0; height:100%; background:rgba(79,195,247,0.25); }}
  .cov-bar-valid {{ position:absolute; top:0; height:100%; background:rgba(129,199,132,0.6); border-radius:3px; }}
  .cov-legend {{ display:flex; gap:12px; margin-top:4px; }}
  .cov-leg-item {{ display:flex; align-items:center; gap:4px; font-size:0.62rem; color:var(--muted); }}
  .cov-swatch {{ width:10px; height:10px; border-radius:2px; flex-shrink:0; }}
  .cov-dates {{ color:var(--text); font-size:0.68rem; }}
  .no-amf-msg {{ color:var(--warn); font-size:0.72rem; }}
  /* overlap status chip */
  .overlap-chip {{
    display:inline-block; padding:1px 7px; border-radius:10px;
    font-size:0.63rem; font-weight:600; font-family:'IBM Plex Mono',monospace;
    margin-left:6px; vertical-align:middle;
  }}
  .chip-ok   {{ background:rgba(129,199,132,0.15); color:var(--ok);   border:1px solid rgba(129,199,132,0.3); }}
  .chip-warn {{ background:rgba(242,139,48,0.15);  color:var(--warn); border:1px solid rgba(242,139,48,0.3); }}
  .chip-none {{ background:rgba(122,132,153,0.12); color:var(--none); border:1px solid rgba(122,132,153,0.2); }}

  .stats-grid {{
    display:grid; grid-template-columns:repeat(auto-fill,minmax(175px,1fr)); gap:7px;
  }}
  .stat-card {{
    background:var(--surface2); border:1px solid var(--border);
    border-radius:6px; padding:9px 11px;
  }}
  .stat-card .var-label {{
    font-size:0.65rem; color:var(--muted); text-transform:uppercase;
    letter-spacing:0.08em; font-family:'IBM Plex Mono',monospace; margin-bottom:5px;
  }}
  .stat-row {{
    display:flex; justify-content:space-between;
    font-family:'IBM Plex Mono',monospace; font-size:0.74rem; margin-bottom:2px;
  }}
  .stat-row .k {{ color:var(--muted); }}
  .d01v {{ color:var(--d01); font-weight:600; }}
  .d02v {{ color:var(--d02); font-weight:600; }}
  .domain-badge {{
    display:inline-block; font-family:'IBM Plex Mono',monospace;
    font-size:0.6rem; font-weight:600; padding:1px 5px; border-radius:3px;
    margin-right:3px; letter-spacing:0.04em;
  }}
  .bd1 {{ background:rgba(79,195,247,0.12); color:var(--d01); border:1px solid rgba(79,195,247,0.25); }}
  .bd2 {{ background:rgba(255,183,77,0.12); color:var(--d02); border:1px solid rgba(255,183,77,0.25); }}

  /* ── time series ── */
  .ts-controls {{
    padding:6px 14px; border-bottom:1px solid var(--border);
    display:flex; gap:6px; flex-wrap:wrap; align-items:center; flex-shrink:0;
  }}
  .var-btn {{
    background:var(--surface2); border:1px solid var(--border); color:var(--muted);
    font-family:'IBM Plex Mono',monospace; font-size:0.66rem;
    padding:3px 9px; border-radius:4px; cursor:pointer; transition:all 0.15s;
  }}
  .var-btn:hover {{ border-color:var(--accent); color:var(--accent); }}
  .var-btn.active {{ background:rgba(79,195,247,0.1); border-color:var(--accent); color:var(--accent); }}
  #ts-chart {{ flex:1; min-height:0; position:relative; }}
  .placeholder {{
    display:flex; align-items:center; justify-content:center;
    width:100%; height:100%; color:var(--muted);
    font-family:'IBM Plex Mono',monospace; font-size:0.78rem;
    flex-direction:column; gap:8px; opacity:0.45;
  }}
  .placeholder::before {{ content:'◎'; font-size:1.6rem; opacity:0.4; }}
</style>
</head>
<body>

<header>
  <h1>NU-WRF × AmeriFlux</h1>
  <span>WRF window: {WRF_START} → {WRF_END} &nbsp;·&nbsp; d01 / d02</span>
  <div class="domain-toggles">
    <label>Domains</label>
    <button class="dom-btn active" id="toggle-d01" data-domain="d01">d01</button>
    <button class="dom-btn" id="toggle-d02" data-domain="d02">d02</button>
  </div>
  <div class="state-filter">
    <label>State</label>
    <select id="state-select"></select>
    <span id="site-count"></span>
  </div>
</header>

<div class="layout">
  <div class="panel" id="map-panel">
    <div class="ph"><div class="dot"></div>Station Map — IGBP Land Cover</div>
    <div id="map-div"></div>
  </div>
  <div class="panel" id="stats-panel">
    <div class="ph"><div class="dot" style="background:var(--accent2)"></div>Site Statistics</div>
    <div id="site-info"><div class="placeholder">Select a station on the map</div></div>
  </div>
  <div class="panel" id="ts-panel">
    <div class="ph"><div class="dot" style="background:var(--d02)"></div>Time Series</div>
    <div class="ts-controls" id="ts-controls"></div>
    <div id="ts-chart"><div class="placeholder">Select a station on the map</div></div>
  </div>
</div>

<script>
const SITE_DATA   = {site_data_json};
const VAR_MAP     = {VAR_MAP_JSON};
const IGBP_COLORS = {IGBP_COLORS_JSON};
const ALL_TRACES  = {map_traces_json};   // all sites, grouped by IGBP
const STATES      = {STATES_JSON};
const MAP_CENTER  = {{ lat: {map_center_lat}, lon: {map_center_lon} }};
const WRF_START   = "{WRF_START}";
const WRF_END     = "{WRF_END}";
const D01_BBOX    = {D01_BBOX_JSON};
const D02_BBOX    = {D02_BBOX_JSON};

// Pre-build a sid→site lookup and sid→trace-index lookup
const ALL_SITES_LIST = Object.values(SITE_DATA);

let activeSite = null;
let activeVar  = VAR_MAP[0].wvar;
let activeState = "All USA";
const activeDomains = new Set(['d01']); // d02 off by default (may not exist)

// ── Domain outline traces ─────────────────────────────────────────────────────
function domainTraces() {{
  const out = [];
  if (activeDomains.has('d01') && D01_BBOX) {{
    out.push({{
      type: 'scattermapbox', mode: 'lines',
      lat: D01_BBOX.lat, lon: D01_BBOX.lon,
      line: {{ color: 'rgba(79,195,247,0.7)', width: 2 }},
      name: 'WRF d01', hoverinfo: 'name',
      showlegend: true,
    }});
  }}
  if (activeDomains.has('d02') && D02_BBOX) {{
    out.push({{
      type: 'scattermapbox', mode: 'lines',
      lat: D02_BBOX.lat, lon: D02_BBOX.lon,
      line: {{ color: 'rgba(255,183,77,0.7)', width: 2 }},
      name: 'WRF d02', hoverinfo: 'name',
      showlegend: true,
    }});
  }}
  return out;
}}

// Wire up domain toggle buttons
document.querySelectorAll('.dom-btn').forEach(btn => {{
  btn.addEventListener('click', () => {{
    const dom = btn.dataset.domain;
    if (activeDomains.has(dom)) {{
      activeDomains.delete(dom);
      btn.classList.remove('active');
    }} else {{
      activeDomains.add(dom);
      btn.classList.add('active');
    }}
    filterAndRedraw();
  }});
}});

// ── State selector ────────────────────────────────────────────────────────────
const sel = document.getElementById('state-select');
STATES.forEach(st => {{
  const opt = document.createElement('option');
  opt.value = opt.textContent = st;
  sel.appendChild(opt);
}});
sel.addEventListener('change', () => {{
  activeState = sel.value;
  filterAndRedraw();
}});

// ── Build map ─────────────────────────────────────────────────────────────────
const mapLayout = {{
  mapbox: {{ style:'carto-positron', center:MAP_CENTER, zoom:3.8 }},
  margin: {{ l:0,r:0,t:0,b:0 }},
  paper_bgcolor:'#0e1117',
  legend: {{
    title:{{ text:'IGBP', font:{{ size:11,color:'#7a8499' }} }},
    bgcolor:'rgba(22,27,39,0.92)', bordercolor:'#2a3348', borderwidth:1,
    font:{{ size:11,color:'#e8eaf0' }},
  }},
  autosize:true,
}};

// Filter traces to current state, keeping IGBP grouping
function filteredTraces(state) {{
  return ALL_TRACES.map(tr => {{
    if (state === 'All USA') return tr;
    const keep = tr.customdata.map((sid,i) => SITE_DATA[sid] && SITE_DATA[sid].state === state ? i : -1).filter(i => i >= 0);
    if (!keep.length) return null;
    return {{
      ...tr,
      lat:        keep.map(i => tr.lat[i]),
      lon:        keep.map(i => tr.lon[i]),
      text:       keep.map(i => tr.text[i]),
      customdata: keep.map(i => tr.customdata[i]),
    }};
  }}).filter(Boolean);
}}

function filterAndRedraw() {{
  const traces = [...domainTraces(), ...filteredTraces(activeState)];
  const visibleSids = traces.flatMap(t => t.customdata);
  document.getElementById('site-count').textContent = visibleSids.length + ' sites';

  // Zoom to state if not "All USA"
  let layoutUpdate = {{}};
  if (activeState !== 'All USA' && visibleSids.length > 0) {{
    const lats = visibleSids.map(s => SITE_DATA[s].lat);
    const lons = visibleSids.map(s => SITE_DATA[s].lon);
    const clat = (Math.min(...lats) + Math.max(...lats)) / 2;
    const clon = (Math.min(...lons) + Math.max(...lons)) / 2;
    const span = Math.max(Math.max(...lats)-Math.min(...lats), Math.max(...lons)-Math.min(...lons));
    const zoom = span < 2 ? 7 : span < 5 ? 5.5 : span < 10 ? 4.5 : 3.8;
    layoutUpdate = {{ 'mapbox.center': {{ lat:clat, lon:clon }}, 'mapbox.zoom': zoom }};
  }} else {{
    layoutUpdate = {{ 'mapbox.center': MAP_CENTER, 'mapbox.zoom': 3.8 }};
  }}

  Plotly.react('map-div', traces, mapLayout, {{ responsive:true, displayModeBar:false }});
  Plotly.relayout('map-div', layoutUpdate);

  // Re-attach click (Plotly.react replaces the element)
  document.getElementById('map-div').on('plotly_click', onMapClick);
}}

Plotly.newPlot('map-div', [...domainTraces(), ...filteredTraces('All USA')], mapLayout, {{ responsive:true, displayModeBar:false }});
document.getElementById('site-count').textContent = ALL_SITES_LIST.length + ' sites';
document.getElementById('map-div').on('plotly_click', onMapClick);

function onMapClick(data) {{
  const pt  = data.points[0];
  const sid = Array.isArray(pt.customdata) ? pt.customdata[0] : pt.customdata;
  if (sid) selectSite(sid);
}}

// ── Var buttons ───────────────────────────────────────────────────────────────
const ctrl = document.getElementById('ts-controls');
ctrl.innerHTML = '<span style="font-size:0.66rem;color:var(--muted);font-family:IBM Plex Mono,monospace;margin-right:4px">Variable:</span>';
VAR_MAP.forEach(v => {{
  const btn = document.createElement('button');
  btn.className = 'var-btn' + (v.wvar === activeVar ? ' active' : '');
  btn.textContent = v.label;
  btn.onclick = () => {{
    activeVar = v.wvar;
    document.querySelectorAll('.var-btn').forEach(b => b.classList.remove('active'));
    btn.classList.add('active');
    if (activeSite) renderTimeSeries(activeSite, activeVar);
  }};
  ctrl.appendChild(btn);
}});

// ── Select site ───────────────────────────────────────────────────────────────
function selectSite(sid) {{
  activeSite = sid;
  renderStats(sid);
  renderTimeSeries(sid, activeVar);
}}

// ── Stats panel ───────────────────────────────────────────────────────────────
function coverageBar(s) {{
  if (!s.has_amf || !s.amf_full_start) {{
    return `<div class="amf-coverage"><div class="no-amf-msg">⚠ No AmeriFlux file found for this station.</div></div>`;
  }}

  // Calculate positions as % of the full AmeriFlux record
  const tFull0 = new Date(s.amf_full_start).getTime();
  const tFull1 = new Date(s.amf_full_end).getTime();
  const tWrf0  = new Date(WRF_START).getTime();
  const tWrf1  = new Date(WRF_END).getTime();
  const span   = tFull1 - tFull0 || 1;

  const wrfL = Math.max(0, (tWrf0 - tFull0) / span * 100);
  const wrfW = Math.min(100 - wrfL, (tWrf1 - tWrf0) / span * 100);

  let overlapChip;
  if (s.amf_overlap_hrs === 0) {{
    overlapChip = `<span class="overlap-chip chip-none">no overlap</span>`;
  }} else if (s.amf_valid_le < 24) {{
    overlapChip = `<span class="overlap-chip chip-warn">⚠ ${{s.amf_valid_le}} valid hrs</span>`;
  }} else {{
    overlapChip = `<span class="overlap-chip chip-ok">✓ ${{s.amf_valid_le}} valid hrs</span>`;
  }}

  return `<div class="amf-coverage">
    <div class="cov-title">AmeriFlux record ${{overlapChip}}</div>
    <div class="cov-dates">${{s.amf_full_start}} → ${{s.amf_full_end}}</div>
    <div class="cov-bar-wrap">
      <div class="cov-bar-full" style="left:0;width:100%"></div>
      <div class="cov-bar-wrf"  style="left:${{wrfL.toFixed(1)}}%;width:${{wrfW.toFixed(1)}}%"></div>
    </div>
    <div class="cov-legend">
      <div class="cov-leg-item"><div class="cov-swatch" style="background:#2a3348;border:1px solid #3a4460"></div>Full AMF record</div>
      <div class="cov-leg-item"><div class="cov-swatch" style="background:rgba(79,195,247,0.3)"></div>WRF window</div>
      <div class="cov-leg-item"><div class="cov-swatch" style="background:rgba(129,199,132,0.6)"></div>Valid LE obs</div>
    </div>
  </div>`;
}}

function renderStats(sid) {{
  const s = SITE_DATA[sid];
  if (!s) return;
  const color = IGBP_COLORS[s.igbp] || '#888';
  const d02ok = s.in_d02;

  let html = `
    <div class="site-name">${{sid}}</div>
    <div class="site-meta">
      <span style="display:inline-block;width:9px;height:9px;border-radius:50%;background:${{color}};margin-right:5px;vertical-align:middle"></span>
      ${{s.igbp}} &nbsp;|&nbsp; ${{s.state}} &nbsp;|&nbsp; ${{s.lat.toFixed(3)}}°N &nbsp;${{s.lon.toFixed(3)}}°E<br>
      <span class="domain-badge bd1">d01</span>
      ${{d02ok ? '<span class="domain-badge bd2">d02</span>' : '<span style="font-size:0.62rem;color:var(--muted)">d02 outside domain</span>'}}
    </div>`;

  html += coverageBar(s);

  // Only show stat cards if there's actual data
  const hasStats = Object.keys(s.stats.d01).length > 0 || Object.keys(s.stats.d02).length > 0;
  if (!hasStats) {{
    html += `<div style="font-family:IBM Plex Mono,monospace;font-size:0.73rem;color:var(--muted);padding:6px 0">
      No overlapping WRF + AmeriFlux data in this window.</div>`;
  }} else {{
    html += '<div class="stats-grid">';
    VAR_MAP.forEach(v => {{
      const d1 = s.stats.d01[v.wvar] || {{}};
      const d2 = s.stats.d02[v.wvar] || {{}};
      const fmt = x => (x==null||isNaN(x)) ? '—' : x.toFixed(2);
      const has2 = d02ok && d2.r != null && !isNaN(d2.r);
      html += `<div class="stat-card">
        <div class="var-label">${{v.label}} <span style="opacity:0.5">(${{v.units}})</span></div>
        <div style="font-size:0.6rem;color:var(--muted);margin-bottom:4px;font-family:IBM Plex Mono,monospace">
          <span class="domain-badge bd1">d01</span>${{has2?'<span class="domain-badge bd2">d02</span>':''}}
        </div>
        <div class="stat-row"><span class="k">r</span>
          <span><span class="d01v">${{fmt(d1.r)}}</span>${{has2?' / <span class="d02v">'+fmt(d2.r)+'</span>':''}}</span>
        </div>
        <div class="stat-row"><span class="k">ubRMSD</span>
          <span><span class="d01v">${{fmt(d1.ubrmsd)}}</span>${{has2?' / <span class="d02v">'+fmt(d2.ubrmsd)+'</span>':''}}</span>
        </div>
        <div class="stat-row"><span class="k">bias</span>
          <span><span class="d01v">${{fmt(d1.bias)}}</span>${{has2?' / <span class="d02v">'+fmt(d2.bias)+'</span>':''}}</span>
        </div>
      </div>`;
    }});
    html += '</div>';
  }}

  document.getElementById('site-info').innerHTML = html;
}}

// ── Time series ───────────────────────────────────────────────────────────────
function renderTimeSeries(sid, wvar) {{
  const s  = SITE_DATA[sid];
  if (!s) return;
  const vm = VAR_MAP.find(v => v.wvar === wvar);
  const traces = [];

  ['d01','d02'].forEach(dom => {{
    const ts = s.timeseries[dom];
    if (!ts) return;
    traces.push({{
      x: ts.time, y: ts['wrf_'+wvar],
      name: 'WRF '+dom.toUpperCase(),
      type:'scatter', mode:'lines',
      line:{{ color: dom==='d01'?'#4fc3f7':'#ffb74d', width:1.5 }},
    }});
  }});

  const ts0 = s.timeseries.d01 || s.timeseries.d02;
  if (ts0 && vm) {{
    traces.push({{
      x: ts0.time, y: ts0['amf_'+vm.avar],
      name:'AmeriFlux', type:'scatter', mode:'lines',
      line:{{ color:'#81c784', width:1.5, dash:'dot' }},
    }});
  }}

  if (!traces.length) {{
    document.getElementById('ts-chart').innerHTML = '<div class="placeholder">No data in WRF window for this station</div>';
    return;
  }}

  const tsDiv = document.getElementById('ts-chart');
  const tsH = tsDiv.getBoundingClientRect().height || 280;
  const layout = {{
    paper_bgcolor:'#161b27', plot_bgcolor:'#0e1117',
    font:{{ color:'#e8eaf0', family:'IBM Plex Mono,monospace', size:10 }},
    height: tsH,
    margin:{{ l:55,r:16,t:10,b:45 }},
    xaxis:{{ gridcolor:'#2a3348', linecolor:'#2a3348', tickfont:{{size:9}} }},
    yaxis:{{
      title:{{ text: vm ? vm.label+' ('+vm.units+')' : '', font:{{size:9}} }},
      gridcolor:'#2a3348', linecolor:'#2a3348',
    }},
    legend:{{
      bgcolor:'rgba(22,27,39,0.85)', bordercolor:'#2a3348', borderwidth:1,
      font:{{size:10}}, x:0, y:1, xanchor:'left', yanchor:'top',
    }},
    hovermode:'x unified',
  }};

  Plotly.purge('ts-chart');
  Plotly.newPlot('ts-chart', traces, layout, {{
    scrollZoom: true,
    displayModeBar: true,
    modeBarButtonsToRemove: ['autoScale2d','lasso2d','select2d','toggleSpikelines'],
    displaylogo: false,
    responsive: true,
  }});
}}
</script>
</body>
</html>"""

out_path = "./outputs/wrf_ameriflux_dashboard.html"
os.makedirs(os.path.dirname(out_path), exist_ok=True)
with open(out_path, "w") as f:
    f.write(html)

print(f"\n✓ Dashboard written to: {out_path}")


✓ Dashboard written to: ./outputs/wrf_ameriflux_dashboard.html
